<a href="https://colab.research.google.com/github/night-marionette/Pneumonia-Triage/blob/main/01_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
OUT_DIR = '/content/drive/MyDrive/PneumoniaTriage'
os.makedirs(OUT_DIR, exist_ok=True)
print("Folder ready:", OUT_DIR)

Folder ready: /content/drive/MyDrive/PneumoniaTriage


In [3]:
!pip install medmnist -q
from medmnist import PneumoniaMNIST

train_ds = PneumoniaMNIST(split="train", download=True)
val_ds   = PneumoniaMNIST(split="val",   download=True)
test_ds  = PneumoniaMNIST(split="test",  download=True)

print("Train:", len(train_ds), "| Val:", len(val_ds), "| Test:", len(test_ds))

100%|██████████| 4.17M/4.17M [00:06<00:00, 644kB/s]


Train: 4708 | Val: 524 | Test: 624


In [7]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# A simple neural network for images
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 2) # 2 classes: Normal or Pneumonia
        )
    def forward(self, x):
        return self.net(x)

# Set up GPU and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

print("Training started...")
for epoch in range(5):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.float().to(device)
        labels = labels.squeeze().long().to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/5 | Loss: {total_loss/len(train_loader):.3f}")

print("Training complete!")

Training started...
Epoch 1/5 | Loss: 0.479
Epoch 2/5 | Loss: 0.236
Epoch 3/5 | Loss: 0.171
Epoch 4/5 | Loss: 0.150
Epoch 5/5 | Loss: 0.144
Training complete!


In [8]:
import pandas as pd
import numpy as np

# 1. Load the test data (no shuffle! We need the original order)
test_ds = PneumoniaMNIST(split="test", download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

model.eval()
all_scores = []
all_labels = []

# 2. Get the scores
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.float().to(device)
        outputs = model(imgs)
        # Get the probability that the image is Pneumonia (class 1)
        probs = torch.softmax(outputs, dim=1)[:, 1]

        all_scores.extend(probs.cpu().numpy())
        all_labels.extend(labels.squeeze().numpy())

all_labels = np.array(all_labels)
all_scores = np.array(all_scores)
preds = (all_scores >= 0.5).astype(int) # 0.5 is our cutoff

# 3. Calculate Baseline vs Model (Judges love this)
base_preds = np.ones_like(all_labels) # Baseline: always guess pneumonia
print("=== BASELINE (Always guess Pneumonia) ===")
print(f"Accuracy: {(base_preds == all_labels).mean()*100:.1f}%")
print(f"Normal recall (healthy kids caught): {(base_preds[all_labels==0] == 0).mean()*100:.1f}%")

print("\n=== OUR MODEL ===")
print(f"Accuracy: {(preds == all_labels).mean()*100:.1f}%")

# Calculate Recall and False Alarms
tp = ((preds==1) & (all_labels==1)).sum()
fn = ((preds==0) & (all_labels==1)).sum()
fp = ((preds==1) & (all_labels==0)).sum()

print(f"Pneumonia recall (sick kids caught): {tp / (tp + fn)*100:.1f}% | Missed (False Negatives): {fn}")
print(f"False alarms (healthy kids flagged): {fp}")

# 4. Save to Drive for Golova
df = pd.DataFrame({
    "id": range(len(all_labels)),
    "true_label": all_labels,
    "score": all_scores
})
save_path = '/content/drive/MyDrive/PneumoniaTriage/test_scores.csv'
df.to_csv(save_path, index=False)
print(f"\nSaved! File is ready for Golova at: {save_path}")

=== BASELINE (Always guess Pneumonia) ===
Accuracy: 62.5%
Normal recall (healthy kids caught): 0.0%

=== OUR MODEL ===
Accuracy: 83.8%
Pneumonia recall (sick kids caught): 97.9% | Missed (False Negatives): 8
False alarms (healthy kids flagged): 93

Saved! File is ready for Golova at: /content/drive/MyDrive/PneumoniaTriage/test_scores.csv


In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [10]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import pandas as pd

# 1. FIX THE SEED (So Golova's numbers never change)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

# 2. SETUP PATHS
DATA_DIR = '/kaggle/input/chest-xray-pneumonia/chest_xray'
OUT_DIR = '/content/drive/MyDrive/PneumoniaTriage'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Looking for data in:", DATA_DIR)
print("Found folders:", os.listdir(DATA_DIR)) # Should say ['test', 'train', 'val']

# 3. TRANSFORMS (ResNet needs 224x224 and 3 channels)
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
test_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# 4. LOAD DATA
full_train = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_tf)
test_ds = datasets.ImageFolder(f'{DATA_DIR}/test', transform=test_tf)

# The Kaggle val folder only has 16 images, so we split the train folder instead
n_val = int(0.15 * len(full_train))
n_tr = len(full_train) - n_val
train_ds, val_ds = random_split(full_train, [n_tr, n_val], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

print(f"Class mapping: {full_train.classes}") # ['NORMAL', 'PNEUMONIA']
print(f"Images -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# 5. LOAD RESNET-18 (Pretrained)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 6. TRAIN
print("\nStarting training (this will take 5-10 minutes)...")
for epoch in range(3):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}/3 | Loss: {running_loss/len(train_loader):.3f} | Val Acc: {100 * correct / total:.1f}%")

# 7. GENERATE FINAL TEST_SCORES.CSV FOR GOLOVA
print("\nGenerating final test scores...")
model.eval()
all_scores = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1] # Probability of Pneumonia
        all_scores.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

df = pd.DataFrame({
    "id": range(len(all_labels)),
    "true_label": all_labels,
    "score": all_scores
})

save_path = f'{OUT_DIR}/test_scores.csv'
df.to_csv(save_path, index=False)
print(f"SUCCESS! Saved {len(df)} rows to {save_path}")

Looking for data in: /kaggle/input/chest-xray-pneumonia/chest_xray
Found folders: ['chest_xray', '__MACOSX', 'val', 'test', 'train']
Class mapping: ['NORMAL', 'PNEUMONIA']
Images -> Train: 4434 | Val: 782 | Test: 624
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]



Starting training (this will take 5-10 minutes)...
Epoch 1/3 | Loss: 0.095 | Val Acc: 97.7%
Epoch 2/3 | Loss: 0.043 | Val Acc: 98.1%
Epoch 3/3 | Loss: 0.022 | Val Acc: 98.5%

Generating final test scores...
SUCCESS! Saved 624 rows to /content/drive/MyDrive/PneumoniaTriage/test_scores.csv
